In [ ]:
# %% Deep learning - Section 19.183
#    Do autoencoders clean Gaussians ?

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.


In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [109]:
# %% Generate data

# 2D Gaussian params
n_per_class = 1000
img_size    = 91
x           = np.linspace(-4,4,img_size)
X,Y         = np.meshgrid(x,x)

# Preallocate tensors for images (N,channels,size,size) and labels (N)
images  = torch.zeros( n_per_class,1,img_size,img_size )
labels  = torch.zeros( n_per_class,3 )

# Generate images
for i in range(n_per_class):

    # Gaussian with random center offset
    c = 2*np.random.randn(2)
    w = np.random.rand()*10 + 5
    G = np.exp( -((X-c[0])**2 + (Y-c[1])**2 ) / w )

    # Layer some noise (base noise = 1/10)
    G = G + np.random.randn(img_size,img_size)/10

    # Add to tensor
    images[i,:,:,:] = torch.tensor(G).view(1,img_size,img_size)
    labels[i,:]     = torch.tensor( [c[0],c[1],w] )


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5))/2
fig,axs = plt.subplots(3,7,figsize=(phi*7,7))

for i,ax in enumerate(axs.flatten()):
    pic = np.random.randint(n_per_class)
    G = np.squeeze( images[pic,:,:] )

    ax.imshow(G,vmin=-1,vmax=1,cmap='jet',extent=[-4,4,-4,4])
    ax.set_title(f'XY=({labels[pic,0]:.0f},{labels[pic,1]:.0f}), W={labels[pic,2]:.0f}')
    ax.plot([-4,4],[0,0],'k--',lw=1)
    ax.plot([0,0],[-4,4],'k--',lw=1)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Noise level = 1/10')
plt.tight_layout()

plt.savefig('figure92_gaussian_parameters.png')
plt.show()
files.download('figure92_gaussian_parameters.png')


In [112]:
# %% Create train and test datasets

# Split data with scikitlearn (10% test data)
train_data,test_data,train_labels,test_labels = train_test_split(images,labels,test_size=0.1)

# Convert to PyTorch datasets
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)

# Convert into DataLoader objects
batch_size   = 16
train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,drop_last=True)
test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])


In [ ]:
# %% Check sizes

# Should be (N,channels,width,height) and (N,3)
print( train_loader.dataset.tensors[0].shape )
print( train_loader.dataset.tensors[1].shape )


In [70]:
# %% Function to generate the model

def gen_model():

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            # Layers with nn.Sequential (activation function is treated as layer)
            self.model = nn.Sequential( nn.Conv2d(1,6,3,padding=1),     # out size: (91+2*1-3)/1 + 1 = 91
                                        nn.ReLU(),
                                        nn.AvgPool2d(2,2),              # out size: 91/2 = 45
                                        nn.Conv2d(6,4,3,padding=1),     # out size: (45+2*1-3)/1 + 1 = 45
                                        nn.ReLU(),
                                        nn.AvgPool2d(2,2),              # out size: 45/2 = 22
                                        nn.Flatten(),                   # vectorise
                                        nn.Linear(22*22*4,50),
                                        nn.Linear(50,3)                 # out size: 3
                                        )

        def forward(self,x):
            return self.model(x)

    # Create model instance
    CNN = Gauss_CNN()

    # Loss function (MSELoss for continous predictions)
    loss_fun = nn.MSELoss()

    # Optimizer
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer


In [ ]:
# %% Test the model on one batch

fc_units = 100
CNN,loss_fun,optimizer = gen_model()

X,y  = next(iter(train_loader))
yHat = CNN(X)

# Check sizes of output and target variable
print()
print(yHat.shape), print()
print(y.shape), print()

# Check loss
loss = loss_fun(yHat,y)
print(loss)


In [ ]:
# %% Check all the parameters in the model

summary(CNN,(1,img_size,img_size))


In [72]:
# %% Function to train the model

def train_model():

    # Parameters, model instance, inizialise vars
    num_epochs = 30
    CNN,loss_fun,optimizer = gen_model()

    train_losses = []
    test_losses  = []

    # Loop over epochs
    for epoch_i in range(num_epochs):

        # Loop over training batches
        batch_loss = []

        for X,y in train_loader:

            # Forward propagation and loss
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Loss and accuracy from this batch
            batch_loss.append(loss.item())

        train_losses.append( np.mean(batch_loss) )

        # Test performance
        CNN.eval()

        with torch.no_grad():
            X,y = next(iter(test_loader))
            yHat = CNN(X)
            loss = loss_fun(yHat,y)

            test_losses.append(loss.item())

        CNN.train()

    return train_losses,test_losses,CNN


In [117]:
# %% Run the model

# Takes ~1 min
train_losses,test_losses,CNN = train_model()


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*6,6))

plt.plot(train_losses,'s-',label='Train')
plt.plot(test_losses,'o-',label='Test')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.title(f'Model loss (final test loss: {test_losses[-1]:.2f})')

plt.savefig('figure93_gaussian_parameters.png')
plt.show()
files.download('figure93_gaussian_parameters.png')


In [ ]:
# %% Plotting

# Data
X,Y = next(iter(test_loader))
yHat = CNN(X)

th = np.linspace(0,2*np.pi)

# Plotting
phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(3,5,figsize=(phi*6,6))

for i,ax in enumerate(axs.flatten()):

    # Draw Gaussian
    G = torch.squeeze( X[i,0,:,:] ).detach()
    ax.imshow(G,vmin=-1,vmax=1,cmap='jet',extent=[-4,4,-4,4],origin='lower')
    ax.plot([-4,4],[0,0],'k--',lw=1)
    ax.plot([0,0],[-4,4],'k--',lw=1)

    # Model prediction (center [x,y] and radius)
    cx = yHat[i][0].item()
    cy = yHat[i][1].item()
    rd = yHat[i][2].item()

    # Draw predictions
    x = cx + np.cos(th)*np.sqrt(rd)
    y = cy + np.sin(th)*np.sqrt(rd)
    ax.plot(x,y,'b')
    ax.plot(cx,cy,'bo')

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim([-4,4])
    ax.set_ylim([-4,4])

plt.tight_layout()

plt.savefig('figure94_gaussian_parameters.png')
plt.show()
files.download('figure94_gaussian_parameters.png')


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*6,6))

param_names = ['$C_x$','$C_y$','$radius$']
theor_x = np.linspace(-7,18,500)
theor_y = theor_x

plt.plot(theor_x,theor_y,'k--',lw=1)

for i in range(3):

    # Get parameters and compute correlation with true values
    yy = Y[:,i].detach()
    yh = yHat[:,i].detach()
    cr = np.corrcoef(yy,yh)[0,1]

    plt.plot(yy,yh,'o',label=f'{param_names[i]}, r={cr:.3f}', alpha=0.6)

plt.legend()
plt.xlim([-6,16])
plt.ylim([-6,16])
plt.xlabel('True values')
plt.ylabel('Predicted values')
plt.title('Model predictions vs true values')
plt.grid()

plt.savefig('figure95_gaussian_parameters.png')
plt.show()
files.download('figure95_gaussian_parameters.png')


In [ ]:
# %% Exercise 1
#    Is this model robust to noise? Explore this by increasing the amount of noise added to each stimulus. You can set
#    this up as a parametric experiment if you want, but you can also do it informally, by changing the gain factor of
#    the noise to see whether performance noticeably declines when the images get noisier. Does changing the noise affect
#    the center coordinates or the width more? And what do the results tell you about the power -- or limitations -- of
#    using CNNs for finding features in images?

# Wow, even for a supervided learning model, it is still quite robust even when
# increasing the noise level from 1/10 to 2; the spread estimation (radius)
# suffers a bit more


In [114]:
# %% Exercise 2
#    You can see from the code that I didn't change the model architecture -- I literally copy/pasted it from the previous
#    Gaussian codes and only added 3 units at the end. Do you think you can develop a different architecture, possibly
#    simpler, that achieves comparable performance while reducing learning time?

# An easy way would be to cut down the FC layers, in this case we can try to
# remove the first one and just have one FC output layer

# Function to generate the model
def gen_model():

    class Gauss_CNN(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential( nn.Conv2d(1,6,3,padding=1),     # out size: (91+2*1-3)/1 + 1 = 91
                                        nn.ReLU(),
                                        nn.AvgPool2d(2,2),              # out size: 91/2 = 45
                                        nn.Conv2d(6,4,3,padding=1),     # out size: (45+2*1-3)/1 + 1 = 45
                                        nn.ReLU(),
                                        nn.AvgPool2d(2,2),              # out size: 45/2 = 22
                                        nn.Flatten(),                   # vectorise
                                        nn.Linear(22*22*4,3)            # out size: 3
                                        )

        def forward(self,x):
            return self.model(x)

    CNN = Gauss_CNN()
    loss_fun = nn.MSELoss()
    optimizer = torch.optim.Adam(CNN.parameters(),lr=0.001)

    return CNN,loss_fun,optimizer
